# Measurements preparations

## Preamble

In [2]:
import os
from glob import glob
from datetime import datetime
import csv
import xml.etree.ElementTree as ET


import pandas as pd
from libnmap.parser import NmapParser
import ipaddress

## IPv4

### NMap allowlist

In [59]:
ts = datetime(2025, 8, 27)  # date when ISI hitlist was made

In [60]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i..."
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '..."
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '..."


13361


In [61]:
ipv4_pdf = pd.read_csv(f"internet_address_verfploeter_hitlist_it113w-{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
ipv4_pdf.columns = ["ipv4"]

display(ipv4_pdf.head(5))
print(len(ipv4_pdf))

,ipv4
0,1.0.0.0
1,1.0.4.1
2,1.0.5.1
3,1.0.6.1
4,1.0.7.1


5788894


In [62]:
ipv4_pdf["prefix"] = ipv4_pdf["ipv4"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")

display(ipv4_pdf.head(5))

,ipv4,prefix
0,1.0.0.0,1.0.0.0/24
1,1.0.4.1,1.0.4.0/24
2,1.0.5.1,1.0.5.0/24
3,1.0.6.1,1.0.6.0/24
4,1.0.7.1,1.0.7.0/24


In [63]:
responsive_ipv4_pdf = anycast_ipv4_pdf.merge(ipv4_pdf, on="prefix", how="inner")

display(responsive_ipv4_pdf.head(5))
print(len(responsive_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations,ipv4
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i...",1.0.0.0
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '...",1.1.1.0
2,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'...",1.10.10.10
3,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id...",1.12.0.0
4,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '...",1.12.12.12


13355


In [ ]:
responsive_ipv4 = responsive_ipv4_pdf.drop_duplicates(subset=["ipv4"])["ipv4"].copy()
responsive_ipv4.to_csv(f"../input/nmap/responsive_anycast_ipv4_addresses_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for shard in range(nshards):
    shard_pdf = responsive_ipv4.iloc[shard::nshards]
    shard_pdf.to_csv(f"../input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# confirming that shards don't overlap

all_ips = set()
for shard in range(nshards):
    shard_pdf = pd.read_csv(f"input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
    shard_pdf.columns = ["ipv4"]
    shard_ips = set(shard_pdf["ipv4"].tolist())
    intersection = all_ips.intersection(shard_ips)
    assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
    all_ips.update(shard_ips)

### ZMap allowlist

In [35]:
ts = datetime(2025, 11, 26)

In [ ]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))
anycast_ipv4_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv", index=False, header=False)

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,28,29,28,66,30,False,1.0.0.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
1,1.1.1.0/24,28,29,24,66,30,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
3,1.10.10.0/24,3,0,3,2,0,False,1.10.10.0/24,148000,"[{'city': 'Delhi', 'country_code': 'IN', 'id':..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203,"[{'city': 'Singapore', 'country_code': 'SG', '..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


14010


#### ZMap ports

In [37]:
xml_files = glob("../results/nmap/*.xml")

ports = set()
for xml_file in xml_files:
    print(f"Processing {xml_file}...")
    report = NmapParser.parse_fromfile(xml_file)
    for host in report.hosts:
        for svc in host.services:
            # tcpwrapped are hosts that deliberately closed the TCP connection
            # https://secwiki.org/w/FAQ_tcpwrapped
            if svc.service in ["tcpwrapped"]:
                continue
            if svc.state == "open":
                ports.add(svc.port)


print(f"Discovered {len(ports)} open ports")

with open("../input/zmap/nmap_open_ports.txt", "w") as f:
    for item in ports:
        f.write(f"{item}\n")

Processing ../results/nmap/nmap_20251202093456_.xml...
Discovered 1997 open ports


In [42]:
# tab delimited file
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
nmap_services_pdf["portnum"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: int(x.split("/")[0]))
nmap_services_pdf["protocol"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: x.split("/")[1])

default_nmap_ports_pdf = nmap_services_pdf[nmap_services_pdf["protocol"] == "tcp"].head(2000).copy()

default_nmap_ports = set(default_nmap_ports_pdf["portnum"].tolist())

scanned_but_missing_default = ports.difference(default_nmap_ports)
print(len(scanned_but_missing_default), "ports from measurements are not in default nmap top 2000 ports:", scanned_but_missing_default)

default_but_missing_ports = default_nmap_ports.difference(ports)
print(len(default_but_missing_ports), "default nmap ports are missing in our scan results:", default_but_missing_ports)

316 ports from measurements are not in default nmap top 2000 ports: {1, 2070, 4161, 2124, 2134, 2142, 4192, 2148, 2150, 2187, 4243, 2197, 2203, 4262, 2224, 2250, 2253, 2261, 2262, 2265, 2269, 2270, 2271, 2280, 2291, 2292, 4342, 2300, 2302, 2304, 4355, 4356, 4357, 4358, 2312, 2313, 2325, 2326, 2330, 2335, 2340, 2371, 2372, 2375, 2376, 2391, 2418, 2425, 2435, 2436, 2438, 2439, 2449, 2456, 2463, 2472, 2550, 2551, 2580, 2583, 2691, 4767, 4770, 2723, 4771, 4778, 4793, 4800, 4819, 2804, 2806, 4859, 2812, 4860, 4876, 4877, 4881, 2847, 2850, 4903, 4912, 2882, 4931, 2888, 2889, 2898, 2901, 2902, 2908, 2930, 2957, 2958, 2973, 2984, 2987, 2988, 2991, 2997, 3002, 3014, 3023, 3057, 3062, 3063, 3080, 3089, 3118, 1101, 1116, 1118, 1125, 1128, 1134, 1135, 1136, 5233, 5234, 5235, 3190, 1143, 1144, 1150, 1153, 5250, 1156, 1157, 5252, 1159, 1162, 5259, 5261, 1167, 1168, 1173, 1176, 1179, 1180, 1182, 1184, 1188, 1190, 1191, 1194, 1195, 1196, 5291, 1200, 1204, 1207, 1208, 1209, 1210, 1211, 1215, 1221, 1223

nmap scans top 1000 ports per protocol: https://nmap.org/book/man-port-specification.html

In [ ]:
def parse_nmap_ports(port_string):
    """
    Parse an Nmap port range string like:
    '1,3-4,6-7,9,13,17-20'
    and return a sorted list of all port numbers.
    """
    ports = set()

    for part in port_string.split(","):
        part = part.strip()
        
        # Range: e.g., "300-305"
        if "-" in part:
            start, end = part.split("-")
            start = int(start)
            end = int(end)
            ports.update(range(start, end + 1))
        
        # Single port
        else:
            ports.add(int(part))

    return set(sorted(ports))


# nmap --top-ports 2000 localhost -v -oG -
result2k = """1,3-4,6-7,9,13,17,19-27,30,32-33,37,42-43,49,53,55,57,59,70,77,79-90,98-100,102,106,109-111,113,119,123,125,127,135,139,143-144,146,157,161,163,179,199,210-212,220,222-223,225,250-252,254-257,259,264,280,301,306,311,333,340,366,388-389,406-407,411,416-417,419,425,427,441-445,447,458,464-465,475,481,497,500,502,512-515,523-524,540-541,543-545,548,554-557,563,587,593,600,602,606,610,616-617,621,623,625,631,636,639,641,646,648,655,657,659-660,666-669,674,683-684,687,690-691,700-701,705,709-711,713-715,720,722,725-726,728-732,740,748-749,754,757-758,765,777-778,780,782-783,786-787,790,792,795,800-803,805-806,808,822-823,825,829,839-840,843,846,856,859,862,864,873-874,878,880,888,898,900-905,911-913,918,921-922,924,928,930-931,943,953,969,971,980-981,987,990,992-993,995-996,998-1002,1004-1015,1020-1114,1116-1119,1121-1128,1130-1132,1134-1138,1141,1143-1145,1147-1154,1156-1159,1162-1169,1173-1176,1179-1180,1182-1188,1190-1192,1194-1196,1198-1201,1204,1207-1213,1215-1218,1220-1223,1228-1229,1233-1234,1236,1239-1241,1243-1244,1247-1251,1259,1261-1262,1264,1268,1270-1272,1276-1277,1279,1282,1287,1290-1291,1296-1297,1299-1303,1305-1311,1314-1319,1321-1322,1324,1327-1328,1330-1331,1334,1336-1337,1339-1340,1347,1350-1353,1357,1413-1414,1417,1433-1434,1443,1455,1461,1494,1500-1501,1503,1516,1521-1522,1524-1526,1533,1547,1550,1556,1558-1560,1565-1566,1569,1580,1583-1584,1592,1594,1598,1600,1605,1607,1615,1620,1622,1632,1635,1638,1641,1645,1658,1666,1677,1683,1687-1688,1691,1694,1699-1701,1703,1707-1709,1711-1713,1715,1717-1723,1730,1735-1736,1745,1750,1752-1753,1755,1761,1782-1783,1791-1792,1799-1801,1805-1808,1811-1812,1823,1825,1835,1839-1840,1858,1861-1864,1871,1875,1883,1900-1901,1911-1912,1914,1918,1924,1927,1935,1947,1954,1958,1971-1976,1981,1984,1998-2013,2020-2022,2025,2030-2031,2033-2035,2038,2040-2049,2062,2065,2067-2070,2080-2083,2086-2087,2095-2096,2099-2101,2103-2107,2111-2112,2115,2119,2121,2124,2126,2134-2135,2142,2144,2148,2150,2160-2161,2170,2179,2187,2190-2191,2196-2197,2200-2201,2203,2222,2224,2232,2241,2250-2251,2253,2260-2262,2265,2269-2271,2280,2288,2291-2292,2300-2302,2304,2312-2313,2323,2325-2326,2330,2335,2340,2366,2371-2372,2375-2376,2381-2383,2391,2393-2394,2399,2401,2418,2425,2433,2435-2436,2438-2439,2449,2456,2463,2472,2492,2500-2501,2505,2522,2525,2531-2532,2550-2551,2557-2558,2567,2580,2583-2584,2598,2600-2602,2604-2608,2622-2623,2628,2631,2638,2644,2691,2700-2702,2706,2710-2712,2717-2718,2723,2725,2728,2734,2800,2804,2806,2809,2811-2812,2847,2850,2869,2875,2882,2888-2889,2898,2901-2903,2908-2910,2920,2930,2957-2958,2967-2968,2973,2984,2987-2988,2991,2997-2998,3000-3003,3005-3007,3011,3013-3014,3017,3023,3025,3030-3031,3050,3052,3057,3062-3063,3071,3077,3080,3089,3102-3103,3118-3119,3121,3128,3146,3162,3167-3168,3190,3200,3210-3211,3220-3221,3240,3260-3261,3263,3268-3269,3280-3281,3283,3291,3299-3301,3304,3306-3307,3310-3311,3319,3322-3325,3333-3334,3351,3362-3363,3365,3367-3372,3374,3376,3388-3390,3396,3399-3400,3404,3410,3414-3415,3419,3425,3430,3439,3443,3456,3476,3479,3483,3485-3486,3493,3497,3503,3505-3506,3511,3513-3515,3517,3519-3520,3526-3527,3530,3532,3546,3551,3577,3580,3586,3599-3600,3602-3603,3621-3622,3632,3636-3637,3652-3653,3656,3658-3659,3663,3669-3670,3672,3680-3681,3683-3684,3689-3690,3697,3700,3703,3712,3728,3731,3737,3742,3749,3765-3766,3784,3787-3788,3790,3792-3793,3795-3796,3798-3801,3803,3806,3808-3814,3817,3820,3823-3828,3830-3831,3837,3839,3842,3846-3853,3856,3859-3860,3863,3868-3872,3876,3878-3880,3882,3888-3890,3897,3899,3901-3902,3904-3909,3911,3913-3916,3918-3920,3922-3923,3928-3931,3935-3937,3940-3941,3943-3946,3948-3949,3952,3956-3957,3961-3964,3967-3969,3971-3972,3975,3979-3983,3986,3989-4007,4009-4010,4016,4020,4022,4024-4025,4029,4035-4036,4039-4040,4045,4056,4058,4065,4080,4087,4090,4096,4100-4101,4111-4113,4118-4121,4125-4126,4129,4135,4141,4143,4147,4158,4161,4164,4174,4190,4192,4200,4206,4220,4224,4234,4242-4243,4252,4262,4279,4294,4297-4298,4300,4302,4321,4325,4328,4333,4342-4343,4355-4358,4369,4374-4376,4384,4388,4401,4407,4414-4415,4418,4430,4433,4442-4447,4449,4454,4464,4471,4476,4516-4517,4530,4534,4545,4550,4555,4558-4559,4567,4570,4599-4602,4606,4609,4644,4649,4658,4662,4665,4687,4689,4700,4712-4713,4745,4760,4767,4770-4771,4778,4793,4800,4819,4848,4859-4860,4875-4877,4881,4899-4900,4903,4912,4931,4949,4998-5005,5009-5017,5020-5021,5023,5030,5033,5040,5050-5055,5060-5061,5063,5066,5070,5074,5080-5081,5087-5088,5090,5095-5096,5098,5100-5102,5111,5114,5120-5122,5125,5133,5137,5147,5151-5152,5190,5200-5202,5212,5214,5219,5221-5223,5225-5226,5233-5235,5242,5250,5252,5259,5261,5269,5279-5280,5291,5298,5339,5347,5353,5357,5370,5377,5405,5414,5423,5431-5433,5440-5442,5444,5457-5458,5500-5501,5510,5520,5544,5550,5555,5560,5566,5631,5633,5666,5678-5680,5718,5730,5800-5803,5807,5810-5812,5815,5818,5822-5823,5825,5850,5859,5862,5868-5869,5877,5899-5907,5909-5911,5914-5915,5918,5922,5925,5938,5940,5950,5952,5959-5963,5968,5981,5985-5989,5998-6009,6017,6025,6050-6051,6059-6060,6068,6100-6101,6103,6106,6112,6123,6129,6156,6203,6222,6247,6346,6389,6481,6500,6502,6504,6510,6520,6543,6547,6550,6565-6567,6580,6600,6646,6662,6666-6670,6689,6692,6699,6711,6732,6779,6788-6789,6792,6839,6881,6896,6901,6969,7000-7004,7007,7010,7019,7024-7025,7050-7051,7070,7080,7100,7103,7106,7123,7200-7201,7241,7272,7278,7281,7402,7435,7438,7443,7496,7512,7625,7627,7676,7725,7741,7744,7749,7770,7777-7778,7800,7878,7900,7911,7913,7920-7921,7929,7937-7938,7999-8002,8007-8011,8015-8016,8019,8021-8022,8031,8042,8045,8050,8080-8090,8093,8095,8097-8100,8118,8180-8181,8189,8192-8194,8200,8222,8254,8290-8294,8300,8333,8383,8385,8400,8402,8443,8481,8500,8540,8600,8648-8649,8651-8652,8654,8675-8676,8686,8701,8765-8766,8800,8873,8877,8888-8889,8899,8987,8994,8996,9000-9003,9009-9011,9040,9050,9071,9080-9081,9090-9091,9098-9103,9110-9111,9152,9191,9197-9198,9200,9207,9220,9290,9409,9415,9418,9443-9444,9485,9500-9503,9535,9575,9593-9595,9600,9618,9621,9643,9666,9673,9815,9876-9878,9898,9900,9914,9917,9929,9941,9943-9944,9968,9988,9992,9998-10005,10008-10012,10022-10025,10034,10058,10082-10083,10160,10180,10215,10243,10566,10616-10617,10621,10626,10628-10629,10778,10873,11110-11111,11967,12000,12006,12021,12059,12174,12215,12262,12265,12345-12346,12380,12452,13456,13722,13724,13782-13783,14000,14238,14441-14442,15000-15004,15402,15660,15742,16000-16001,16012,16016,16018,16080,16113,16705,16800,16851,16992-16993,17595,17877,17988,18000,18018,18040,18101,18264,18988,19101,19283,19315,19350,19780,19801,19842,19900,20000,20002,20005,20031,20221-20222,20828,21571,21792,22222,22939,23052,23502,23796,24444,24800,25734-25735,26000,26214,26470,27000,27352-27353,27355-27357,27715,28201,28211,29672,29831,30000,30005,30704,30718,30951,31038,31337,31727,32768-32785,32791-32792,32803,32816,32822,32835,33354,33453,33554,33899,34571-34573,35500,35513,37839,38037,38185,38188,38292,39136,39376,39659,40000,40193,40811,40911,41064,41511,41523,42510,44176,44334,44442-44443,44501,44709,45100,46200,46996,47544,48080,49152-49161,49163-49165,49167-49168,49171,49175-49176,49186,49195,49236,49400-49401,49999-50003,50006,50050,50300,50389,50500,50636,50800,51103,51191,51413,51493,52660,52673,52710,52735,52822,52847-52851,52853,52869,53211,53313-53314,53535,54045,54328,55020,55055-55056,55555,55576,55600,56737-56738,57294,57665,57797,58001-58002,58080,58630,58632,58838,59110,59200-59202,60020,60123,60146,60443,60642,61532,61613,61900,62078,63331,64623,64680,65000,65129,65310,65389"""

localhost_ports = parse_nmap_ports(result2k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

print(len(localhost_ports.difference(ports)))

Nmap default top ports count: 2000
3


In [46]:
result1k = """1,3-4,6-7,9,13,17,19-26,30,32-33,37,42-43,49,53,70,79-85,88-90,99-100,106,109-111,113,119,125,135,139,143-144,146,161,163,179,199,211-212,222,254-256,259,264,280,301,306,311,340,366,389,406-407,416-417,425,427,443-445,458,464-465,481,497,500,512-515,524,541,543-545,548,554-555,563,587,593,616-617,625,631,636,646,648,666-668,683,687,691,700,705,711,714,720,722,726,749,765,777,783,787,800-801,808,843,873,880,888,898,900-903,911-912,981,987,990,992-993,995,999-1002,1007,1009-1011,1021-1100,1102,1104-1108,1110-1114,1117,1119,1121-1124,1126,1130-1132,1137-1138,1141,1145,1147-1149,1151-1152,1154,1163-1166,1169,1174-1175,1183,1185-1187,1192,1198-1199,1201,1213,1216-1218,1233-1234,1236,1244,1247-1248,1259,1271-1272,1277,1287,1296,1300-1301,1309-1311,1322,1328,1334,1352,1417,1433-1434,1443,1455,1461,1494,1500-1501,1503,1521,1524,1533,1556,1580,1583,1594,1600,1641,1658,1666,1687-1688,1700,1717-1721,1723,1755,1761,1782-1783,1801,1805,1812,1839-1840,1862-1864,1875,1900,1914,1935,1947,1971-1972,1974,1984,1998-2010,2013,2020-2022,2030,2033-2035,2038,2040-2043,2045-2049,2065,2068,2099-2100,2103,2105-2107,2111,2119,2121,2126,2135,2144,2160-2161,2170,2179,2190-2191,2196,2200,2222,2251,2260,2288,2301,2323,2366,2381-2383,2393-2394,2399,2401,2492,2500,2522,2525,2557,2601-2602,2604-2605,2607-2608,2638,2701-2702,2710,2717-2718,2725,2800,2809,2811,2869,2875,2909-2910,2920,2967-2968,2998,3000-3001,3003,3005-3006,3011,3017,3030-3031,3052,3071,3077,3128,3168,3211,3221,3260-3261,3268-3269,3283,3300-3301,3306,3322-3325,3333,3351,3367,3369-3372,3389-3390,3404,3476,3493,3517,3527,3546,3551,3580,3659,3689-3690,3703,3737,3766,3784,3800-3801,3809,3814,3826-3828,3851,3869,3871,3878,3880,3889,3905,3914,3918,3920,3945,3971,3986,3995,3998,4000-4006,4045,4111,4125-4126,4129,4224,4242,4279,4321,4343,4443-4446,4449,4550,4567,4662,4848,4899-4900,4998,5000-5004,5009,5030,5033,5050-5051,5054,5060-5061,5080,5087,5100-5102,5120,5190,5200,5214,5221-5222,5225-5226,5269,5280,5298,5357,5405,5414,5431-5432,5440,5500,5510,5544,5550,5555,5560,5566,5631,5633,5666,5678-5679,5718,5730,5800-5802,5810-5811,5815,5822,5825,5850,5859,5862,5877,5900-5904,5906-5907,5910-5911,5915,5922,5925,5950,5952,5959-5963,5985-5989,5998-6007,6009,6025,6059,6100-6101,6106,6112,6123,6129,6156,6346,6389,6502,6510,6543,6547,6565-6567,6580,6646,6666-6669,6689,6692,6699,6779,6788-6789,6792,6839,6881,6901,6969,7000-7002,7004,7007,7019,7025,7070,7100,7103,7106,7200-7201,7402,7435,7443,7496,7512,7625,7627,7676,7741,7777-7778,7800,7911,7920-7921,7937-7938,7999-8002,8007-8011,8021-8022,8031,8042,8045,8080-8090,8093,8099-8100,8180-8181,8192-8194,8200,8222,8254,8290-8292,8300,8333,8383,8400,8402,8443,8500,8600,8649,8651-8652,8654,8701,8800,8873,8888,8899,8994,9000-9003,9009-9011,9040,9050,9071,9080-9081,9090-9091,9099-9103,9110-9111,9200,9207,9220,9290,9415,9418,9485,9500,9502-9503,9535,9575,9593-9595,9618,9666,9876-9878,9898,9900,9917,9929,9943-9944,9968,9998-10004,10009-10010,10012,10024-10025,10082,10180,10215,10243,10566,10616-10617,10621,10626,10628-10629,10778,11110-11111,11967,12000,12174,12265,12345,13456,13722,13782-13783,14000,14238,14441-14442,15000,15002-15004,15660,15742,16000-16001,16012,16016,16018,16080,16113,16992-16993,17877,17988,18040,18101,18988,19101,19283,19315,19350,19780,19801,19842,20000,20005,20031,20221-20222,20828,21571,22939,23502,24444,24800,25734-25735,26214,27000,27352-27353,27355-27356,27715,28201,30000,30718,30951,31038,31337,32768-32785,33354,33899,34571-34573,35500,38292,40193,40911,41511,42510,44176,44442-44443,44501,45100,48080,49152-49161,49163,49165,49167,49175-49176,49400,49999-50003,50006,50300,50389,50500,50636,50800,51103,51493,52673,52822,52848,52869,54045,54328,55055-55056,55555,55600,56737-56738,57294,57797,58080,60020,60443,61532,61900,62078,63331,64623,64680,65000,65129,65389"""

localhost_ports = parse_nmap_ports(result1k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

with open("../input/zmap/nmap_open_ports.txt", "w") as f:
    for item in localhost_ports:
        f.write(f"{item}\n")

Nmap default top ports count: 1000


### ZGrab2 allowlist

In [5]:
def store_zgrab_input(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_files = glob(f"../results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_file in zmap_files:
        # extracting the port from the file name
        port = int(zmap_file.split("_")[-2])
        zmap_port_pdf = pd.read_csv(zmap_file)
        zmap_port_pdf["port"] = port
        zmap_pdf = pd.concat([zmap_pdf, zmap_port_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_pdf["domain"] = ""
    zmap_pdf["tag"] = ""
    zmap_pdf[["saddr", "domain", "tag", "port"]].to_csv(f"../input/zgrab/zgrab_input_{dataset}.csv", index=False, header=False)

store_zgrab_input("tcp-anycast")

27


### NMap allowlist -- old pipeline

In [44]:
def create_allowlist_from_zmap(dataset, ts, output_filename):
    # zmap v4 files are like: zmap_465_20251127123220_v4.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_v4_pdf.drop_duplicates(["saddr"])["saddr"].to_csv(f"input/nmap/{output_filename}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
ts = datetime(2025, 11, 28)  # date when ZMap scan was done
create_allowlist_from_zmap("tcp-anycast", ts, "zmap_responsive_tcp_ipv4_addresses")

### QUIC allowlist

In [45]:
ts = datetime(2025, 11, 28)  # date when ZMap UDP scan was done
create_allowlist_from_zmap("udp-anycast", ts, "zmap_responsive_udp_ipv4_addresses")

## IPv6

### ZMap allowlist -- unused

In [2]:
ts_v6 = datetime(2025, 11, 26)

In [ ]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))
anycast_ipv6_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv", index=False, header=False)

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201::/48,2,2,2,2,1,2001:1201::/48,28498,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
11,2001:1258::/48,2,2,2,2,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
13,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', '..."
14,2001:12f8:4::/48,4,0,4,5,0,2001:12f8:4::/48,11644,"[{'city': None, 'country_code': None, 'id': 'N..."
15,2001:12f8:8::/48,3,0,3,3,0,2001:12f8:8::/48,10906,"[{'city': 'St Petersburg', 'country_code': 'RU..."


12821


### NMap allowlist

In [9]:
ts_v6 = datetime(2025, 8, 18)

In [10]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '..."
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
6,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
8,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
10,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '..."


6019


In [11]:
ipv6_pdf = pd.read_csv(f"v6_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)

ipv6_pdf.columns = ["ipv6"]
display(ipv6_pdf.head(5))
print(len(ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


6234585


In [12]:
def is_v6_in_prefix(ipv6, prefix):
    try:
        ipv6_addr = ipaddress.IPv6Address(ipv6)
        ipv6_net = ipaddress.IPv6Network(prefix)
        return ipv6_addr in ipv6_net
    except ValueError:
        return False


def ipv6_to_prefix48(ip):
    try:
        # Create a /48 network that contains this IP, non-strict so host bits allowed
        net = ipaddress.IPv6Network(f"{ip}/48", strict=False)
        # Return in same format as your prefix column, e.g. "2001:1201:10::/48"
        return net.with_prefixlen
    except ValueError:
        return None


ipv6_pdf["prefix"] = ipv6_pdf["ipv6"].apply(lambda x: ipv6_to_prefix48(x))
display(ipv6_pdf.head(5))

,ipv6,prefix
0,2620:10a:80ac::45,2620:10a:80ac::/48
1,2001:678:2c:0:194:0:28:7,2001:678:2c::/48
2,2001:678:78:42:ad::53,2001:678:78::/48
3,2001:dce:2000:2::130,2001:dce:2000::/48
4,2001:dce:7000:2::130,2001:dce:7000::/48


In [13]:
responsive_ipv6_pdf = anycast_ipv6_pdf.merge(ipv6_pdf, on="prefix", how="inner")

display(responsive_ipv6_pdf.head(5))
print(len(responsive_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations,ipv6
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '...",2001:1201:10::1
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1201::1
2,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:a000::1
3,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:c000::1
4,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '...",2001:1258::1


6017


In [14]:
responsive_ipv6_pdf["ipv6"].to_csv(f"input/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", index=False, header=False)

# Process output

## Process ZMap output

In [ ]:
def process_zmap_anycast_results(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    display(
        zmap_v4_pdf.groupby("port").size().reset_index(name="counts").sort_values(by="counts", ascending=False)
    )

    print("unique reponsive IPv4 addresses:", zmap_v4_pdf["saddr"].nunique())

process_zmap_anycast_results("tcp-anycast")

,port,counts
8,443,2428483
4,80,2376009
3,53,683857
5,110,623389
14,995,619655
13,993,619218
1,22,614430
2,25,610767
19,3306,609978
6,143,609429


unique reponsive IPv4 addresses: 2598785


In [ ]:
process_zmap_anycast_results("udp-anycast")

,port,counts
0,443,1053948
1,853,1327


unique reponsive IPv4 addresses: 1053974


## Process nmap output

In [57]:
xml_files = glob("results/nmap/*.xml")

for xml_file in xml_files:
    print(f"Processing {xml_file}...")

    # parse rtt for each host
    tree = ET.parse(xml_file)
    root = tree.getroot()

    host_rtts = {}
    for host in root.findall("host"):
        addr_el = host.find("address")
        ip = addr_el.get("addr") if addr_el is not None else None

        times = host.find("times")
        if times is not None:
            srtt = int(times.get("srtt")) / 1000   # convert to ms
            host_rtts[ip] = srtt

    report = NmapParser.parse_fromfile(xml_file)
    filename = os.path.splitext(os.path.basename(xml_file))[0]
    with open(f"results/nmap/{filename}.csv", "wt", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["host", "rtt_ms", "port", "protocol", "service", "os", "banner"])
        for host in report.hosts:
            #print(dir(host))
            for svc in host.services:
                writer.writerow([
                    host.address,
                    host_rtts.get(host.address),
                    svc.port,
                    svc.protocol,
                    svc.service,
                    host.os,
                    svc.banner,
                ])



Processing results/nmap/nmap_20251127155539_.xml...
Processing results/nmap/nmap_20251128195856_.xml...
Processing results/nmap/nmap_20251128203007_.xml...
Processing results/nmap/output.xml...
Processing results/nmap/nmap_20251128193545_.xml...


In [21]:
# https://secwiki.org/w/FAQ_tcpwrapped

In [58]:
nmap_pdf = pd.read_csv("results/nmap/nmap_20251128203007_.csv")
display(nmap_pdf)
print(len(nmap_pdf))

nmap_pdf.groupby(["host", "service"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)

,host,rtt_ms,port,protocol,service,os,banner


0


,host,service,counts


In [49]:
nmap_pdf = pd.read_csv("results/nmap/output.csv")
display(nmap_pdf)
print(len(nmap_pdf))

,host,rtt_ms,port,protocol,service,os,banner
0,1.0.0.0,1.212,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
1,1.0.0.0,1.212,443,tcp,https,Fingerprints:,product: cloudflare
2,1.0.0.0,1.212,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
3,1.0.0.0,1.212,8443,tcp,https-alt,Fingerprints:,product: cloudflare
4,1.1.1.0,1.250,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
5,1.1.1.0,1.250,443,tcp,https,Fingerprints:,product: cloudflare
6,1.1.1.0,1.250,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
7,1.1.1.0,1.250,8443,tcp,https-alt,Fingerprints:,product: cloudflare
8,1.1.8.8,207.842,80,tcp,http,Fingerprints:,product: nginx version: 1.20.1
9,1.1.8.8,207.842,443,tcp,http,Fingerprints:,product: OpenResty web app server


24


## Process mtr output

In [34]:
mtr_files = glob("results/mtr/*.csv")
for mtr_file in mtr_files:
    print(f"Processing {mtr_file}...")
    # Add your processing code here
    mtr_pdf = pd.read_csv(mtr_file)
    display(mtr_pdf)

Processing results/mtr/mtr_1.12.15.31.csv...


,hostname,Mtr_Version,Start_Time,Status,Host,Hop,Ip,Asn,Loss%,Snt,,Last,Avg,Best,Wrst,StDev
0,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,1,172.18.0.1 (172.18.0.1),AS???,0.0,1,0,0.33,0.33,0.33,0.33,0.0
1,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,2,169.254.169.254 (169.254.169.254),AS???,0.0,1,0,0.20,0.20,0.20,0.20,0.0
2,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,3,vl199-ds1-j2-51917.ams1.constant.com (45.76.40...,AS20473,0.0,1,0,18.60,18.60,18.60,18.60,0.0
3,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,4,10.72.0.5 (10.72.0.5),AS???,0.0,1,0,1.78,1.78,1.78,1.78,0.0
4,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,5,10.72.4.13 (10.72.4.13),AS???,0.0,1,0,0.95,0.95,0.95,0.95,0.0
5,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,6,5-1-40.ear3.amsterdam1.level3.net (213.19.196....,AS3356,0.0,1,0,1.63,1.63,1.63,1.63,0.0
6,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,7,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
7,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,8,62.67.67.70 (62.67.67.70),AS3356,0.0,1,0,8.88,8.88,8.88,8.88,0.0
8,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,9,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
9,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,10,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
